# [Part 2 Lab] Data-driven MLP 기반 SOC 추정

온도(`T`), 전류(`I`), 전압(`V`) **한 행(row)** 을 입력하면 같은 step의 SOC(%)를 출력하는 간단한 MLP 회귀 실습입니다.

> **데이터 경계**
>
> - 전류·기준 SOC·OCV–SOC 표: 기존 `EKF_SOC_estimation` 예제 파일
> - 온도: 원본에 온도 채널이 없어 1차 열응답 모델로 합성
> - MLP 입력 전압: `OCV(SOC) + I·R0(T)`와 작은 센서 잡음으로 교육용 합성
> - 원본 DST 전압은 CSV의 `source_voltage_V` 열에 비교용으로 보존


## 학습 목표

1. `temperature_C`, `current_A`, `voltage_V`를 입력 특징으로 구성한다.
2. 같은 시간 블록의 온도 변형이 train/test에 동시에 들어가지 않도록 시간 블록 단위로 분할한다.
3. `StandardScaler → MLP(3 → 32 → 16 → 1)`을 학습한다.
4. MAE, RMSE, $R^2$와 시간 이력으로 결과를 읽는다.
5. `estimate_soc(T, I, V)`로 한 step SOC를 추정한다.

In [ ]:
from pathlib import Path
import json
import platform

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260729
FEATURES = ["temperature_C", "current_A", "voltage_V"]
TARGET = "soc_pct"

OUTPUT_DIR = Path.cwd() / "soc_mlp_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.unicode_minus": False,
})

print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)
print("Output directory:", OUTPUT_DIR.resolve())

## 1. 데이터 불러오기

노트북과 함께 제공된 `soc_mlp_dst_temperature_dataset.csv`를 Colab `/content`에 업로드하세요.
로컬에서 실행할 때는 패키지의 `02_data` 폴더도 자동으로 검색합니다.

In [ ]:
DATA_NAME = "soc_mlp_dst_temperature_dataset.csv"
candidate_paths = [
    Path(DATA_NAME),
    Path("02_data") / DATA_NAME,
    Path("/content") / DATA_NAME,
]
data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    try:
        from google.colab import files
        print(f"{DATA_NAME} 파일을 선택하세요.")
        uploaded = files.upload()
        if DATA_NAME not in uploaded:
            raise FileNotFoundError(f"{DATA_NAME}가 업로드되지 않았습니다.")
        data_path = Path("/content") / DATA_NAME
    except ImportError as exc:
        raise FileNotFoundError(
            f"{DATA_NAME}를 현재 폴더 또는 02_data 폴더에 놓아주세요."
        ) from exc

df = pd.read_csv(data_path)
required = set(FEATURES + [TARGET, "split", "scenario_id", "base_step", "block_id"])
missing = sorted(required.difference(df.columns))
if missing:
    raise ValueError(f"필수 열이 없습니다: {missing}")

print("Data:", data_path.resolve())
print("Shape:", df.shape)
display(df.head())

## 2. 입력과 정답을 확인하기

- 입력 `X`: 온도, 전류, 전압 3개
- 정답 `y`: 같은 step의 기준 SOC(%)
- 음의 전류는 기존 EKF SOC 예제와 동일하게 방전 방향입니다.

In [ ]:
display(df[FEATURES + [TARGET]].describe().T)

sample_scenario = df[df["scenario_id"] == "ambient_+25.0C"].sort_values("base_step")
fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
for ax, column, ylabel in zip(
    axes,
    ["temperature_C", "current_A", "voltage_V", "soc_pct"],
    ["Temperature [°C]", "Current [A]", "Voltage [V]", "SOC [%]"],
):
    ax.plot(sample_scenario["base_step"], sample_scenario[column], linewidth=1.0)
    ax.set_ylabel(ylabel)
axes[-1].set_xlabel("Base time step")
plt.tight_layout()
plt.show()

## 3. 시간 블록 단위 분할

행을 무작위로 섞으면 바로 이웃한 step이나 같은 step의 다른 온도 변형이 양쪽에 섞일 수 있습니다.
이 예제는 `block_id` 전체를 train/validation/test 중 하나에만 배정합니다.

In [ ]:
split_summary = (
    df.groupby("split")
      .agg(rows=("soc_pct", "size"), blocks=("block_id", "nunique"),
           soc_min=("soc_pct", "min"), soc_max=("soc_pct", "max"))
      .reindex(["train", "validation", "test"])
)
display(split_summary)

block_check = df.groupby("block_id")["split"].nunique()
assert int(block_check.max()) == 1, "한 block이 여러 split에 들어갔습니다."
print("누수 점검 PASS: 각 block은 하나의 split에만 존재합니다.")

## 4. 전처리와 MLP 구성

`StandardScaler`는 **train 데이터에만 fit**됩니다. Pipeline 안에서 학습하면 같은 규칙이 추론에도 자동 적용됩니다.
SOC 정답은 학습 안정성을 위해 0~1 fraction으로 나눈 뒤, 출력할 때 다시 100을 곱합니다.

구조: `3 inputs → 32 ReLU → 16 ReLU → 1 linear output`

In [ ]:
train_df = df[df["split"] == "train"].copy()
validation_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_validation, y_validation = validation_df[FEATURES], validation_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

model = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        batch_size=128,
        learning_rate_init=1e-3,
        max_iter=500,
        shuffle=True,
        random_state=SEED,
        early_stopping=True,
        validation_fraction=0.10,
        tol=1e-5,
        n_iter_no_change=30,
    )),
])

model.fit(X_train, y_train / 100.0)
print(model)
print("Iterations:", model.named_steps["mlp"].n_iter_)

## 5. 학습 상태 확인

손실이 줄어드는지 확인합니다. 고정된 반복 횟수 내에서 충분히 수렴하지 않았다면 `max_iter`나 학습률을 조정할 수 있습니다.

In [ ]:
loss_curve = model.named_steps["mlp"].loss_curve_
plt.figure(figsize=(8, 3.8))
plt.plot(np.arange(1, len(loss_curve) + 1), loss_curve, linewidth=2)
plt.xlabel("Iteration")
plt.ylabel("Training loss (half MSE, SOC fraction)")
plt.title("MLP training loss")
plt.tight_layout()
plt.show()

## 6. 성능 평가

- MAE, RMSE 단위는 SOC **%-point**
- $R^2$는 1에 가까울수록 좋습니다.
- validation은 모델 선택 연습용, test는 마지막 보고용입니다.

In [ ]:
def regression_metrics(y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred), 0.0, 100.0)
    y_true = np.asarray(y_true)
    return {
        "MAE [%-point]": mean_absolute_error(y_true, y_pred),
        "RMSE [%-point]": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "Max |error| [%-point]": np.max(np.abs(y_pred - y_true)),
    }

rows = []
predictions = {}
for split_name, X_part, y_part in [
    ("train", X_train, y_train),
    ("validation", X_validation, y_validation),
    ("test", X_test, y_test),
]:
    pred = np.clip(model.predict(X_part) * 100.0, 0.0, 100.0)
    predictions[split_name] = pred
    rows.append({"split": split_name, **regression_metrics(y_part, pred)})

metrics_df = pd.DataFrame(rows).set_index("split")
display(metrics_df.round(4))

## 7. 미사용 test 시간 블록의 SOC 이력

한 온도 시나리오에서 test 블록만 골라 기준 SOC와 MLP 출력을 겹쳐 봅니다.
블록 사이의 빈 구간은 학습/검증에 배정된 시간입니다.

In [ ]:
test_result = test_df.copy()
test_result["soc_estimated_pct"] = predictions["test"]
test_result["residual_pct_point"] = (
    test_result["soc_estimated_pct"] - test_result["soc_pct"]
)

timeline = test_result[
    test_result["scenario_id"] == "ambient_+25.0C"
].sort_values("base_step")

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(timeline["base_step"], timeline["soc_pct"], "k-", label="Reference SOC")
axes[0].plot(
    timeline["base_step"], timeline["soc_estimated_pct"],
    color="#ED7D31", linestyle="--", label="MLP estimate"
)
axes[0].set_ylabel("SOC [%]")
axes[0].legend(ncol=2)
axes[1].plot(
    timeline["base_step"], timeline["residual_pct_point"],
    color="#7030A0"
)
axes[1].axhline(0, color="k", linewidth=0.8)
axes[1].set_ylabel("Error [%-point]")
axes[1].set_xlabel("Base time step")
plt.tight_layout()
plt.show()

## 8. 한 step SOC 추정 함수

아래 함수가 요청한 최종 형태입니다. 온도·전류·전압 스칼라 3개를 넣으면 해당 step의 SOC(%) 하나가 나옵니다.

In [ ]:
def estimate_soc(temperature_C, current_A, voltage_V):
    one_step = pd.DataFrame([{
        "temperature_C": float(temperature_C),
        "current_A": float(current_A),
        "voltage_V": float(voltage_V),
    }])
    soc_estimated = float(model.predict(one_step)[0] * 100.0)
    return float(np.clip(soc_estimated, 0.0, 100.0))

example = timeline.iloc[len(timeline) // 2]
soc_hat = estimate_soc(
    example["temperature_C"],
    example["current_A"],
    example["voltage_V"],
)

print(f"Input T = {example['temperature_C']:.2f} °C")
print(f"Input I = {example['current_A']:.3f} A")
print(f"Input V = {example['voltage_V']:.3f} V")
print(f"Estimated SOC = {soc_hat:.2f} %")
print(f"Reference SOC = {example['soc_pct']:.2f} %")

## 9. 모델 저장

`Pipeline` 전체를 저장하므로 StandardScaler와 MLP가 함께 보존됩니다.

In [ ]:
model_path = OUTPUT_DIR / "soc_mlp_model.joblib"
metrics_path = OUTPUT_DIR / "soc_mlp_metrics.json"
prediction_path = OUTPUT_DIR / "soc_mlp_test_predictions.csv"

joblib.dump(model, model_path)
metrics_path.write_text(
    json.dumps(metrics_df.reset_index().to_dict(orient="records"), indent=2),
    encoding="utf-8",
)
test_result.to_csv(prediction_path, index=False)

print("Saved model:", model_path.resolve())
print("Saved metrics:", metrics_path.resolve())
print("Saved predictions:", prediction_path.resolve())

## EKF 예제와 연결해서 이해하기

| 항목 | 기존 EKF SOC 예제 | 이번 MLP SOC 예제 |
|---|---|---|
| 핵심 지식 | OCV-SOC, 1RC ECM, 잡음 공분산 | 라벨된 입력-출력 데이터 |
| step 입력 | 전류 + 측정 전압 | 온도 + 전류 + 전압 |
| 내부 상태 | SOC, RC 분지 전류 | 은닉층 활성값 |
| 출력 | SOC와 모델 전압 | SOC |
| 장점 | 물리 해석과 순차 보정 | 학습 후 계산이 단순하고 빠름 |
| 주의점 | 모델/파라미터 오차 | 학습 범위 밖 일반화와 데이터 누수 |

둘 중 하나가 항상 우월한 것이 아니라, 데이터 범위와 검증 조건에 맞게 선택해야 합니다.

## 한계와 다음 실습

- 온도는 합성 채널이므로 실제 온도 센서 데이터로 다시 학습해야 합니다.
- test는 같은 DST 궤적의 미사용 시간 블록입니다. 다른 셀·다른 주행 프로파일 검증이 필요합니다.
- 단일 step MLP는 시간 이력을 직접 기억하지 않습니다.
- 추가 실습: 은닉층 크기 변경, 온도 제거 비교, 센서 잡음 증가, 완전히 다른 drive cycle 평가.